# Phase 4B — Correctness and reduced-index ranking quality

**Question 1 (exact correctness):** On the **same normalized MiniLM 384d vectors**, do float32 BLAS cosine scores agree with an independently computed float64 dot-product reference within **absolute tolerance 1e-5**? Are top-10 IDs and their ordering identical? Score correctness and ranking correctness are **separate tests**; near ties can cause rank differences despite correct scores.

**Question 2 (approximate ranking quality):** What fraction of the exact 384d top-10 document IDs are recovered by the fitted 100d TruncatedSVD index? This is *top-10 overlap*, not ground-truth relevance accuracy. **Do not apply 1e-5 to reduced vs original cosine scores**, because dimensionality reduction changes the geometry.

Uses real Qwen 20K embeddings and the **already fitted Phase 4A SVD model**. Pilot: 1K docs, 12 in-corpus queries. Full: 20K docs, 100 in-corpus queries plus 10 external text queries. Same deterministic query IDs are used for each method. Save raw per-query results, detailed discrepancies, summary and figure. No search timings are presented as performance benchmarks here; scaling comes in 4C.

In [ ]:
%pip -q install sentence-transformers scikit-learn joblib pandas matplotlib threadpoolctl
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, platform, time, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from threadpoolctl import threadpool_limits

ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
OUT=ROOT/'phase4_final_system'
PILOT=True  # Change to False after reviewing pilot; rerun ALL cells.
N=1000 if PILOT else 20000
NUM_IN_CORPUS=12 if PILOT else 100
NUM_EXTERNAL=0 if PILOT else 10
TOP_K=10
ABS_TOL=1e-5
SEED=20261021
TAG='pilot_1k' if PILOT else 'full_20k'
(OUT/'benchmarks').mkdir(parents=True,exist_ok=True)
(OUT/'figures').mkdir(parents=True,exist_ok=True)
print('Mode:',TAG,'N:',N,'corpus queries:',NUM_IN_CORPUS,'external:',NUM_EXTERNAL)

## 1. Load the exact and reduced indexes from Phase 4A

For fairness, **both indexes contain the same first N document IDs**. The exact index is row-normalized float32 MiniLM. The reduced index is the saved, row-normalized 100d SVD representation. We reuse the fitted model, not retrain it. Query normalization follows Phase 4A exactly.

In [ ]:
assert EMB.exists(), f'Missing cached embedding file: {EMB}'
E=np.load(EMB,mmap_mode='r')
assert E.shape==(20000,384) and E.dtype==np.float32,(E.shape,E.dtype)
X=np.ascontiguousarray(normalize(np.asarray(E[:N],dtype=np.float32),axis=1),dtype=np.float32)
svd_path=OUT/'models'/f'truncated_svd_k100_{TAG}.joblib'
z_path=OUT/'models'/f'reduced_index_k100_{TAG}.npy'
assert svd_path.exists() and z_path.exists(), 'First finish Phase 4A for this PILOT/FULL mode.'
svd=joblib.load(svd_path)
Z=np.ascontiguousarray(np.load(z_path),dtype=np.float32)
assert Z.shape==(N,100) and np.isfinite(Z).all()
assert np.allclose(np.linalg.norm(X,axis=1),1.0,atol=1e-5)
assert np.allclose(np.linalg.norm(Z,axis=1),1.0,atol=1e-5)
print('Loaded X:',X.shape,'Z:',Z.shape,'model:',svd_path.name)

## 2. Prepare deterministic queries

**In-corpus queries** use document embeddings sampled without replacement (reproducible seed). The **full run** also embeds 10 newly written, external text queries with the same MiniLM model as the index; this checks that the query transform is actually usable. In-corpus queries can artificially inflate overlap because a document matches itself: we report corpus and external queries **separately**. No claim of ground-truth relevance is made.

In [ ]:
rng=np.random.default_rng(SEED)
sample_ids=rng.choice(N,size=NUM_IN_CORPUS,replace=False)
query_rows=[]
for doc_id in sample_ids:
    q_raw=np.asarray(E[int(doc_id)],dtype=np.float32)
    query_rows.append(('in_corpus',f'doc_{int(doc_id)}',q_raw))
external_texts=[
    'How can machine learning improve cybersecurity threat detection?',
    'Explain the risks of phishing and social engineering.',
    'What is the difference between supervised and unsupervised learning?',
    'How do electric vehicles affect the energy grid?',
    'Describe effective ways to reduce software vulnerabilities.',
    'Why is renewable energy important for climate change?',
    'What are the ethical risks of generative artificial intelligence?',
    'How can doctors use data science for early disease detection?',
    'Explain how database indexes improve query performance.',
    'What are the benefits and drawbacks of cloud computing?'
]
if NUM_EXTERNAL:
    encoder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device='cpu')
    external_vectors=encoder.encode(external_texts[:NUM_EXTERNAL],convert_to_numpy=True,normalize_embeddings=False)
    assert external_vectors.shape==(NUM_EXTERNAL,384)
    for text,vec in zip(external_texts[:NUM_EXTERNAL],external_vectors):
        query_rows.append(('external',text,np.asarray(vec,dtype=np.float32)))
print('Total queries:',len(query_rows),'types:',pd.Series([x[0] for x in query_rows]).value_counts().to_dict())

## 3. Reference vs BLAS: independent score calculation and top-k

- **Reference**: float64 accumulation with `X.astype(float64) @ q.astype(float64)` on already normalized float32 vectors. This tests numerical arithmetic, not normalization accuracy.
- **Optimized**: float32 BLAS `X @ q` with one BLAS thread.
- Rank by decreasing score, break ties by smaller document ID (`np.lexsort`). Compute score error for **all N documents** per query, not only top-10.
- Save both exact top-10 **set match** and **ordered match**; near-ties can change order even if max score error is below tolerance. No automatic claim that all rankings must match.
- Compare SVD top-10 ID overlap against the float64 reference. Exact vs reduced score magnitudes are not directly comparable.

In [ ]:
def rank(scores,k=TOP_K):
    ids=np.lexsort((np.arange(len(scores)), -scores))[:k]
    return ids.astype(int)

rows=[]
discrepancies=[]
with threadpool_limits(limits=1):
    X64=np.asarray(X,dtype=np.float64)
    for idx,(q_type,q_label,q_raw) in enumerate(query_rows):
        q=np.ascontiguousarray(normalize(q_raw.reshape(1,-1),axis=1)[0],dtype=np.float32)
        q64=q.astype(np.float64)
        ref_scores=X64@q64
        blas_scores=X@q
        q100=np.ascontiguousarray(normalize(svd.transform(q_raw.reshape(1,-1)),axis=1)[0],dtype=np.float32)
        svd_scores=Z@q100
        assert np.isfinite(ref_scores).all() and np.isfinite(blas_scores).all() and np.isfinite(svd_scores).all()
        ref_ids=rank(ref_scores)
        blas_ids=rank(blas_scores)
        svd_ids=rank(svd_scores)
        abs_error=np.abs(ref_scores-blas_scores)
        max_err=float(abs_error.max())
        bad_scores=int(np.count_nonzero(abs_error>ABS_TOL))
        ref_set=set(ref_ids.tolist())
        blas_set=set(blas_ids.tolist())
        svd_set=set(svd_ids.tolist())
        set_match=ref_set==blas_set
        order_match=np.array_equal(ref_ids,blas_ids)
        overlap=len(ref_set&svd_set)/TOP_K
        rows.append(dict(query_index=idx,query_type=q_type,query_label=q_label,
                         n_docs=N,dim_exact=384,dim_reduced=100,
                         max_abs_score_error=max_err,mean_abs_score_error=float(abs_error.mean()),
                         num_scores_over_tolerance=bad_scores,all_scores_within_tolerance=bad_scores==0,
                         exact_top10_set_match=set_match,exact_top10_order_match=order_match,
                         svd_top10_overlap=overlap,
                         reference_top10_ids=' '.join(map(str,ref_ids)),
                         blas_top10_ids=' '.join(map(str,blas_ids)),
                         svd_top10_ids=' '.join(map(str,svd_ids))))
        if bad_scores or not set_match or not order_match:
            # Diagnose rank-boundary and near-tie effects, never hide discrepancies.
            ref_sorted=np.sort(ref_scores)[::-1]
            margin=float(ref_sorted[TOP_K-1]-ref_sorted[TOP_K]) if N>TOP_K else float('nan')
            discrepancies.append(dict(query_index=idx,query_type=q_type,query_label=q_label,
                num_scores_over_tolerance=bad_scores,max_abs_score_error=max_err,
                exact_top10_set_match=set_match,exact_top10_order_match=order_match,
                reference_rank10_rank11_margin=margin,
                reference_top10_ids=' '.join(map(str,ref_ids)),
                blas_top10_ids=' '.join(map(str,blas_ids))))
        if (idx+1)%20==0 or idx+1==len(query_rows):
            print(f'Checked {idx+1}/{len(query_rows)} queries')
results=pd.DataFrame(rows)
detail=pd.DataFrame(discrepancies,columns=[
    'query_index','query_type','query_label','num_scores_over_tolerance',
    'max_abs_score_error','exact_top10_set_match','exact_top10_order_match',
    'reference_rank10_rank11_margin','reference_top10_ids','blas_top10_ids'])
print('Done; mismatched queries:',len(detail))

## 4. Summarize and save full evidence

The instructor's `1e-5` tolerance applies to **reference vs optimized exact scores**, not SVD's different scores. We report ranking changes separately. In full mode, use 100 in-corpus and 10 external queries; medians, minimum, mean overlap and mismatch counts are shown separately for each group. Pilot results must never be described as full-run findings.

In [ ]:
summary=results.groupby('query_type',as_index=False).agg(
    queries=('query_index','count'),
    max_abs_score_error=('max_abs_score_error','max'),
    mean_abs_score_error=('mean_abs_score_error','mean'),
    score_tolerance_pass_rate=('all_scores_within_tolerance','mean'),
    exact_top10_set_match_rate=('exact_top10_set_match','mean'),
    exact_top10_order_match_rate=('exact_top10_order_match','mean'),
    mean_svd_top10_overlap=('svd_top10_overlap','mean'),
    median_svd_top10_overlap=('svd_top10_overlap','median'),
    min_svd_top10_overlap=('svd_top10_overlap','min'))
print(summary.to_string(index=False))
print('Total score tolerance violations:',int(results.num_scores_over_tolerance.sum()))
print('Exact set mismatches:',int((~results.exact_top10_set_match).sum()),
      '| exact order mismatches:',int((~results.exact_top10_order_match).sum()))
print('WARNING: in-corpus overlap is not unbiased retrieval quality; inspect external-query overlap separately.')
raw_path=OUT/'benchmarks'/f'phase4b_correctness_raw_{TAG}.csv'
summary_path=OUT/'benchmarks'/f'phase4b_correctness_summary_{TAG}.csv'
detail_path=OUT/'benchmarks'/f'phase4b_exact_ranking_discrepancies_{TAG}.csv'
results.to_csv(raw_path,index=False)
summary.to_csv(summary_path,index=False)
detail.to_csv(detail_path,index=False)
print('Saved:',raw_path.name,summary_path.name,detail_path.name)

## 5. Visualize ranking overlap by query type

This figure is a **quality diagnostic**, not a speedup graph. Phase 4C will measure scaling with 5 timed repetitions per method/configuration.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
groups=[g for _,g in results.groupby('query_type',sort=False)]
labels=[g.query_type.iloc[0] for g in groups]
values=[g.svd_top10_overlap.to_numpy() for g in groups]
ax.boxplot(values,tick_labels=labels,showmeans=True)
ax.set_ylim(-0.02,1.05)
ax.set_ylabel('Top-10 ID overlap: SVD 100d vs exact 384d')
ax.set_title(f'Phase 4B — SVD ranking agreement ({TAG}; {len(results)} queries)')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig_path=OUT/'figures'/f'phase4b_svd_top10_overlap_{TAG}.png'
fig.savefig(fig_path,dpi=170)
plt.show()
print('Saved:',fig_path)

## Interpretation checklist

- Report **max absolute score error** and **number of tolerance failures** for exact 384d reference vs BLAS.
- Report **top-10 set match and ordered match separately**, including any near-tie discrepancies.
- Report **100d SVD top-10 overlap** as an approximate-ranking measure, **not** accuracy against relevance labels.
- Treat corpus-self-query and external-text-query statistics separately.
- This notebook **does not** benchmark search speed or corpus scaling. **Phase 4C** will test N=1K/5K/10K/20K, 100 shared queries, warmup and 5 timed repetitions, and archive CSV + scaling chart.
- Send the completed Colab outputs for verification before switching from PILOT to FULL.